# 05 — Bidirectional Pincer Search & Dual Oracle Validation
**Analytics and Systems of Big Data Practice — Endsem Hackathon**
*IIITDM Kancheepuram*

## 1. Theoretical Architecture of Pincer Search (Lin & Kedem 1998)
Bidirectional search: Bottom-up candidate generation ($C_k, L_k$) and top-down candidate reduction ($MFCS$).

In [1]:
import sys, os
from pathlib import Path
ROOT = Path(os.getcwd()).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import *
from IPython.display import Image, display

from src.pincer_search import PincerSearch, validate_pincer_against_oracles
import json
with open(PROCESSED_DATA_DIR / 'transactions_predictive_S&P.json') as f:
    tx = json.load(f)
df_bin = pd.read_parquet(PROCESSED_DATA_DIR / 'binary_c2_S&P.parquet')
pincer = PincerSearch(tx, min_support=0.10)
mfs = pincer.fit()
print('Discovered MFS Count:', len(mfs))

Discovered MFS Count: 444


## 2. Level-Wise Iteration Logs & MFCS Antichain Pruning
Inspect the progress of bottom-up $L_k$ and top-down $MFCS$ at each pass.

In [2]:
df_iter = pd.read_csv(TABLES_DIR / 'table_11_pincer_iterations_S&P.csv')
df_iter

,Iteration_k,C_k_Count,L_k_Count,S_k_Count,MFCS_Count,MFS_Count
0,1,24,24,0,1,0
1,2,276,242,34,264,0
2,3,1402,525,877,67,72
3,4,739,235,504,27,362
4,5,44,14,30,0,457


## 3. Dual-Oracle Mathematical Validation
Verify 100% exact set identity between Pincer Search, mlxtend.fpmax, and maximal FP-Growth.

In [3]:
df_oracle = pd.read_csv(TABLES_DIR / 'table_13_oracle_comparison_S&P.csv')
df_oracle

,Algorithm,MFS_Count,Is_Antichain,Matches_Ground_Truth
0,Pincer_Search,444,True,True
1,mlxtend_fpmax,444,True,True
2,fpgrowth_maximal_filter,444,True,True
